In [23]:
import re
import json
import sqlite3
import uuid
from pathlib import Path
import aiosqlite
import httpx
import gradio as gr
from openai import AsyncOpenAI

In [24]:
from google.colab import userdata

In [25]:
openAI_API_KEY = userdata.get('OPENAI_API')

In [26]:
#Setting up client
client = AsyncOpenAI(api_key=openAI_API_KEY)

In [27]:
BASE_PATH = "/content/drive/MyDrive/Colab Notebooks/"

In [28]:
DB = f"{BASE_PATH}/chats.db"

In [29]:
IMG_DIR = Path(f"{BASE_PATH}/OpenAI_Images")
IMG_DIR.mkdir(exist_ok=True)

In [30]:
#Table Setup
with sqlite3.connect(DB) as conn:
  conn.execute("""
  CREATE TABLE IF NOT EXISTS conversations(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    messages TEXT NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
  )
  """)

In [31]:
IMG_RE = re.compile(r"!\[[^\]]*\]\(([^)]+)\)")

In [32]:
#Helper Functions
async def download_image(url:str)->Path:
  path = IMG_DIR/f"{uuid.uiid4().hex}.png"
  async with httpx.AsyncClient(timeout=60) as http:
    r = await http.get(url)
    r.raise_for_status()
  path.write_bytes(r.content)
  return path


In [33]:
def purge_image(history:list[dict])->None:
  for msg in history:
    content = msg.get("content")
    if not isinstance(content, str):
      continue
    for ref in IMG_RE.findall(content):
      p = Path(ref)
      try:
        if p.resolve().is_relative_to(IMG_DIR.resolve()):
          p.unlink(missing_ok=True)
      except (ValueEror, OSError):
        ... #Path is outside our directory or already deleted

In [34]:
#Chat Handler
async def chat(message, history):
  if not message.strip():
    return history, ""

  if message.lower().startswith("/image "):
    prompt = message[7:].strip()
    result = await client.images.generate(
        model="dall-e-3", prompt=prompt, size="1024x1024"
    )
    local_path = await download_image(result.data[0].url)
    reply = f"![{prompt}]({local_path.as_posix()})"
  else:
    msgs = history + [{"role":"user", "content":message}]
    response = await client.chat.completions.create(
       model="gptt-5.4-mini", messages=msgs
    )
    reply = response.choices[0].message.content

  history = history + [
      {"role":"user", "content":message},
      {"role":"assistant", "content":reply}
  ]
  return history, ""

In [41]:
async def list_conversations():
  async with aiosqlite.connect(DB) as db:
    cursor = await db.execute(
      "SELECT id, title FROM conversations ORDER BY created_at DESC"
    )
    rows = await cursor.fetchall()
  # Return gr.update to properly update the existing Dropdown component
  return gr.update(choices=[(r[0], r[1]) for r in rows], value=None)

In [36]:
async def save_conversation(history, title):
  if not history:
    return await list_conversations(), "Nothing to save"
  title = (title or "").strip() or f"Chat - {len(history)//2} turns"
  async with aiosqlite.connect(DB) as db:
    await db.execute(
        "INSERT INTO conversations (title, messages) VALUES (?, ?)", # Corrected table name
        (title, json.dumps(history))
    )
    await db.commit()
  return await list_conversations(), f"Saved: {title}"

In [37]:
async def load_conversation(conv_id):
  if conv_id is None:
    return [], "Select a conversations first"
  async with aiosqlite.connect(DB) as db:
    cursor = await db.execute(
        "SELECT title, messages FROM conversations WHERE id=?", (conv_id,) # Corrected table name
    )
    row = await cursor.fetchone()
  if row is None:
    return [], "Not found"
  return json.loads(row[1]), f"Loaded: {row[0]}"

In [38]:
async def delete_conversation(conv_id):
  if conv_id is None:
    return await list_conversations(), "Select a conversations first"

  async with aiosqlite.connect(DB) as db:
    cursor = await db.execute(
        "SELECT messages FROM conversations WHERE id=?", (conv_id,) # Corrected table name
    )
    row = await cursor.fetchone()
    if row:
      purge_image(json.loads(row[0]))

    await db.execute("DELETE FROM conversation WHERE id=?", (conv_id,)) # Corrected table name
    await db.commit()
  return await list_conversations(), "Deleted"

In [42]:
#UI
with gr.Blocks(title="OpenAI Chatbot") as demo:
  gr.Markdown("# OpenAI Chatbot\nChat normally, or `/image <prompt>` for DALL-E")

  with gr.Row():
    with gr.Column(scale=3):
      chatbot = gr.Chatbot(type="messages", height=500)
    msg = gr.Textbox(placeholder="Message or /image <prompt>...", show_label=False)

    with gr.Column(scale=1):
      gr.Markdown("### Saved Chats")
      saved = gr.Dropdown(label="Conversations", choices=[], interactive=True)
      title_in = gr.Textbox(label="Title (for saving)", placeholder="optional")
      with gr.Row():
        save_btn = gr.Button("Save")
        load_btn = gr.Button("Load")
      with gr.Row():
        del_btn = gr.Button("Delete", variant="stop")
        clear_btn = gr.Button("New Chat")
      status = gr.Markdown("")

  msg.submit(chat, [msg, chatbot], [chatbot, msg])
  save_btn.click(save_conversation, [chatbot, title_in], [saved, status])
  load_btn.click(load_conversation, [saved], [chatbot, status])
  del_btn.click(delete_conversation, [saved], [saved, status])
  clear_btn.click(lambda: ([], ""), outputs=[chatbot, status])
  demo.load(list_conversations, outputs=[saved])

/tmp/ipykernel_702/851418897.py:7: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(type="messages", height=500)


In [44]:
demo.launch(allowed_paths=[str(IMG_DIR.resolve())])

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://27c432901a9bf6e5ce.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
